# Dự đoán rủi ro trễ hạn thẻ tín dụng

Notebook này trình bày Project 08 theo luồng báo cáo. Nó **chỉ đọc** dữ liệu, pipeline đã đóng băng và kết quả test đã lưu; không huấn luyện lại, không chọn lại ngưỡng và không chạy lại test.

## 1. Mục tiêu
Dự đoán xác suất khách hàng có nhãn `default_payment_next_month = 1` (default tháng kế tiếp) để hỗ trợ ưu tiên nhắc thanh toán. Đây là bài thực hành học máy, không phải công cụ quyết định cấp tín dụng.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image
ROOT = Path.cwd()
FREEZE = json.loads((ROOT/'models/final_selection_freeze.json').read_text(encoding='utf-8'))
TEST = json.loads((ROOT/'reports/final_test_evaluation.json').read_text(encoding='utf-8'))
assert FREEZE['selected_model'] == 'logistic_unweighted'
assert FREEZE['selected_C'] == 1.0 and FREEZE['calibration'].startswith('none')
assert FREEZE['threshold'] == 0.15 and TEST['test_evaluation_count'] == 1
print('Cấu hình đóng băng đã kiểm tra:', FREEZE['selected_model'], '| C=', FREEZE['selected_C'], '| ngưỡng=', FREEZE['threshold'])

## 2. Mô tả dữ liệu
Dữ liệu gốc chính thức UCI trong dự án là `data/raw/default of credit card clients.xls`. Nếu bạn đặt `UCI_Credit_Card.csv` vào `data/raw/`, ô dưới ưu tiên đọc CSV để trình bày. CSV Kaggle chỉ là tệp đối chiếu; nguồn huấn luyện được ghi nhận là UCI.

In [ ]:
csv_path = ROOT/'data/raw/UCI_Credit_Card.csv'
xls_path = ROOT/'data/raw/default of credit card clients.xls'
if csv_path.exists():
    raw = pd.read_csv(csv_path)
    print('Đang đọc CSV:', csv_path)
elif xls_path.exists():
    raw = pd.read_excel(xls_path, header=1)
    print('Chưa có UCI_Credit_Card.csv trong data/raw; đang dùng tệp UCI chính thức .xls để mô tả.')
else:
    raise FileNotFoundError('Đặt UCI_Credit_Card.csv vào data/raw hoặc khôi phục tệp UCI .xls.')
raw.columns = [str(c).strip() for c in raw.columns]
display(raw.head())
print('Số dòng, cột:', raw.shape, '| Missing:', int(raw.isna().sum().sum()))
print('Target:', 'default_payment_next_month; ID bị loại khỏi đặc trưng.')

## 3. Khám phá dữ liệu
EDA chỉ được thực hiện trên train để không rò rỉ thông tin từ validation/test. PAY_0, PAY_2 đến PAY_6 là trạng thái trả nợ lịch sử: số dương là số tháng chậm; -1 trả đúng hạn; -2 không tiêu dùng; 0 quay vòng tín dụng.

In [ ]:
train = pd.read_csv(ROOT/'data/processed/train.csv')
validation = pd.read_csv(ROOT/'data/processed/validation.csv')
test = pd.read_csv(ROOT/'data/processed/test.csv')
display(pd.DataFrame({'Tập':['Train','Validation','Test'], 'Số dòng':[len(train),len(validation),len(test)], 'Tỷ lệ default':[train.default_payment_next_month.mean(), validation.default_payment_next_month.mean(), test.default_payment_next_month.mean()]}))
display(Image(filename=str(ROOT/'reports/figures/train_target_distribution.png')))
display(Image(filename=str(ROOT/'reports/figures/train_continuous_feature_correlation.png')))
print('Hai biểu đồ trên chỉ dùng train.')

## 4. Tiền xử lý và chia dữ liệu
Split stratified 60/20/20, `random_state=42`. Pipeline: biến liên tục được median imputation rồi StandardScaler; biến phân loại được most-frequent imputation rồi OneHotEncoder. ID và target bị loại. Pipeline được fit trên train (và riêng trong mỗi CV fold).

In [ ]:
pd.DataFrame({'Nhóm':['Liên tục','Phân loại','Loại bỏ'], 'Chi tiết':[FREEZE['preprocessing']['continuous'], FREEZE['preprocessing']['categorical'], ', '.join(FREEZE['preprocessing']['excluded_columns'])]})

## 5. Huấn luyện và chọn ngưỡng trên validation
Mô hình đã chốt là Logistic Regression không `class_weight`, `C=1.0`; không calibration, giữ raw probabilities. Không chạy lại huấn luyện trong notebook. Chi phí giả định: TN=0, TP=0, FP=1, FN=5. Ngưỡng 0.150 được chọn bằng tối thiểu hóa chi phí trên validation, trước khi mở test.

In [ ]:
display(pd.DataFrame(FREEZE['validation_candidate_costs'])[['model','threshold','accuracy','precision_default','recall_default','pr_auc_average_precision','brier_score','total_assumed_cost']])
display(Image(filename=str(ROOT/'reports/figures/validation_confusion_matrix_threshold_0.150.png')))
print('Ngưỡng chính thức:', FREEZE['threshold'], '| Chi phí validation:', FREEZE['validation_candidate_costs'][0]['total_assumed_cost'])

## 6. Kết quả test đã đóng băng
Test đã được đánh giá đúng **một lần** lúc lưu `final_test_evaluation.json`. Các số dưới đây được đọc từ file kết quả, không gọi model để dự đoán lại. Average Precision (AP) được báo riêng với PR-AUC hình thang.

In [ ]:
m = TEST['metrics']
result = pd.DataFrame([{'Accuracy':m['accuracy'],'Precision default':m['precision_default'],'Recall default':m['recall_default'],'F1':m['f1_default'],'ROC-AUC':m['roc_auc'],'Average Precision':m['pr_auc_average_precision'],'Brier':m['brier_score'],'Tổng chi phí':m['total_assumed_cost'],'Chi phí/khách hàng':m['assumed_cost_per_customer']}])
display(result.style.format(precision=4))
display(pd.DataFrame([[m['tn'],m['fp']],[m['fn'],m['tp']]], index=['Thực tế 0','Thực tế 1'], columns=['Dự đoán 0','Dự đoán 1']))
display(Image(filename=str(ROOT/'reports/figures/final_test_pr_roc_curves.png')))
display(Image(filename=str(ROOT/'reports/figures/final_test_reliability_diagram.png')))
display(Image(filename=str(ROOT/'reports/figures/final_test_group_audit_rates.png')))

## 7. Nhận xét khi trình bày
- Recall test 0.734 nghĩa là phát hiện khoảng 73.4% default, nhưng precision 0.354 cho thấy có nhiều cảnh báo nhầm.
- Accuracy 0.645 ở ngưỡng 0.150 thấp hơn tại 0.5 vì mô hình chủ động đánh đổi accuracy để giảm FN trong kịch bản chi phí giả định.
- AP 0.525 và ROC-AUC 0.759 mô tả khả năng xếp hạng; Brier 0.139 và reliability diagram mô tả chất lượng xác suất.
- Audit nhóm chỉ là mô tả; không chứng minh công bằng hoặc quan hệ nhân quả. Nhóm ít mẫu phải được diễn giải thận trọng.
- Còn thiếu nếu cần nộp notebook hoàn toàn tự chạy bằng CSV: đặt đúng `data/raw/UCI_Credit_Card.csv`. Notebook hiện vẫn chạy mô tả bằng tệp UCI `.xls` có sẵn.